# Travel Insurance Claim Classification

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `claim`

## 2 · Project Overview

We predict whether a **travel insurance policyholder will file a claim** based on traveler demographics, trip details, destination risk, plan type, and claims history.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a traveler's age, trip duration, destination risk, plan type, premium, chronic conditions, number of travelers, and previous claims, predict whether they will file a claim.

## 5 · Why This Project Matters

- **Insurance pricing** depends on accurate claim probability estimation.
- Identifying high-risk policies helps with reserve management.
- Understanding claim drivers improves underwriting decisions.
- Classic imbalanced binary classification in the insurance domain.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | age, trip_duration_days, destination_risk, plan_type, premium_usd, has_chronic_condition, num_travelers, prev_claims |
| **Target** | claim (0 = no claim, 1 = claim filed) |
| **Class balance** | ~75/25 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "claim"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: claim
Save dir: E:\Github\Machine-Learning-Projects\Classification\Travel Insurance Claim Classification


## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 travel insurance policies with claim outcome.

In [4]:
np.random.seed(SEED)
n = 8000
age = np.random.randint(18, 75, n)
trip_duration_days = np.random.randint(1, 30, n)
destination_risk = np.random.choice(["Low", "Medium", "High"], n, p=[0.5, 0.35, 0.15])
dest_num = np.where(destination_risk == "Low", 0, np.where(destination_risk == "Medium", 1, 2))
plan_type = np.random.choice(["Basic", "Standard", "Premium"], n, p=[0.4, 0.35, 0.25])
plan_num = np.where(plan_type == "Basic", 0, np.where(plan_type == "Standard", 1, 2))
premium_usd = np.round(20 + 5 * trip_duration_days + 10 * dest_num + 15 * plan_num
                        + 0.3 * age + np.random.normal(0, 10, n), 2).clip(10, 500)
has_chronic_condition = np.random.binomial(1, 0.15, n)
num_travelers = np.random.randint(1, 6, n)
prev_claims = np.random.poisson(0.3, n).clip(0, 5)

score = (0.02 * age + 0.03 * trip_duration_days + 0.4 * dest_num
         + 0.5 * has_chronic_condition + 0.15 * prev_claims
         - 0.1 * plan_num + 0.05 * num_travelers
         + np.random.normal(0, 1.0, n))
claim = (score > np.percentile(score, 75)).astype(int)

df = pd.DataFrame({"age": age, "trip_duration_days": trip_duration_days,
                    "destination_risk": destination_risk, "plan_type": plan_type,
                    "premium_usd": premium_usd, "has_chronic_condition": has_chronic_condition,
                    "num_travelers": num_travelers, "prev_claims": prev_claims,
                    "claim": claim})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['claim'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (8000, 9)
Class balance:
claim
0    0.75
1    0.25
Name: proportion, dtype: float64


,age,trip_duration_days,destination_risk,plan_type,premium_usd,has_chronic_condition,num_travelers,prev_claims,claim
0,56,3,High,Basic,78.02,0,2,0,0
1,69,1,Medium,Premium,72.35,0,3,0,0
2,46,24,Medium,Standard,180.86,0,3,0,0
3,32,19,High,Premium,170.05,0,3,1,1
4,60,18,Low,Basic,152.01,0,1,0,1


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (8000, 9)

Missing values:
age                      0
trip_duration_days       0
destination_risk         0
plan_type                0
premium_usd              0
has_chronic_condition    0
num_travelers            0
prev_claims              0
claim                    0
dtype: int64

Duplicate rows: 1

Dtypes:
age                        int32
trip_duration_days         int32
destination_risk          object
plan_type                 object
premium_usd              float64
has_chronic_condition      int32
num_travelers              int32
prev_claims                int32
claim                      int64
dtype: object

Target 'claim' confirmed. Value counts:
claim
0    6000
1    2000
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["age", "trip_duration_days", "premium_usd", "num_travelers", "prev_claims"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
axes[1][2].axis("off")
plt.suptitle("Feature Distributions by Claim Status", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.groupby("destination_risk")[TARGET].mean().reindex(["Low", "Medium", "High"]).plot(
    kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Claim Rate by Destination Risk")
df.groupby("has_chronic_condition")[TARGET].mean().plot(
    kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Claim Rate by Chronic Condition")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `claim`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "salmon"], edgecolor="black")
ax.set_title("Travel Insurance Claim Distribution")
ax.set_xticklabels(["No Claim (0)", "Claim (1)"], rotation=0)
plt.tight_layout()
plt.show()
print(f"Claim rate: {df[TARGET].mean():.1%}")

Claim rate: 25.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (6400, 8), Test: (1600, 8)
Train class distribution:
claim
0    0.75
1    0.25
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `destination_risk`, `plan_type` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: ~25% claims (imbalanced).

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["risk_age_interaction"] = X_train["age"] * X_train["destination_risk"]
X_test["risk_age_interaction"] = X_test["age"] * X_test["destination_risk"]

X_train["premium_per_day"] = X_train["premium_usd"] / (X_train["trip_duration_days"] + 1)
X_test["premium_per_day"] = X_test["premium_usd"] / (X_test["trip_duration_days"] + 1)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (10): ['age', 'trip_duration_days', 'destination_risk', 'plan_type', 'premium_usd', 'has_chronic_condition', 'num_travelers', 'prev_claims', 'risk_age_interaction', 'premium_per_day']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.7588
F1       : 0.2548

              precision    recall  f1-score   support

           0       0.77      0.96      0.86      1200
           1       0.56      0.17      0.25       400

    accuracy                           0.76      1600
   macro avg       0.67      0.56      0.56      1600
weighted avg       0.72      0.76      0.71      1600



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
NearestCentroid                0.617500           0.627500  0.672090  0.642863   0.717147  0.617500    0.026107
QuadraticDiscriminantAnalysis  0.745000           0.622500  0.707321  0.733446   0.727007  0.745000    0.029671
GaussianNB                     0.736875           0.619583  0.704252  0.727842   0.721860  0.736875    0.018488
XGBClassifier                  0.733125           0.586250  0.664983  0.712370   0.703555  0.733125    0.395846
AdaBoostClassifier             0.763750           0.585833  0.714939  0.724383   0.731336  0.763750    0.246879
KNeighborsClassifier           0.732500           0.579167  0.651617  0.708680   0.699583  0.732500    0.128618
DecisionTreeClassifier         0.669375           0.572917  0.572917  

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.6931
F1: 0.3356


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.2346  (3.5s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[39]	valid_0's binary_logloss: 0.510779
LightGBM F1: 0.1796  (2.1s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                  0.6931  0.3356     0.3658  0.3100
Logistic Regression    0.7588  0.2548     0.5593  0.1650
CatBoost               0.7512  0.2346     0.5083  0.1525
LightGBM               0.7488  0.1796     0.4889  0.1100

Top 3 models: ['FLAML', 'Logistic Regression', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  FLAML:
    Accuracy : 0.6931
    F1       : 0.3356
    Precision: 0.3658
    Recall   : 0.3100

  Logistic Regression:
    Accuracy : 0.7588
    F1       : 0.2548
    Precision: 0.5593
    Recall   : 0.1650

  CatBoost:
    Accuracy : 0.7512
    F1       : 0.2346
    Precision: 0.5083
    Recall   : 0.1525


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: FLAML

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.82      0.80      1200
           1       0.37      0.31      0.34       400

    accuracy                           0.69      1600
   macro avg       0.57      0.57      0.57      1600
weighted avg       0.68      0.69      0.68      1600


Total errors: 491 / 1600 (30.7%)

Sample misclassifications:
      age  trip_duration_days  destination_risk  plan_type  premium_usd  has_chronic_condition  num_travelers  prev_claims  risk_age_interaction  premium_per_day  actual  predicted  correct
7885   36                  25               1.0        2.0       150.67                      0              3            0                  36.0         5.795000       1          0    False
3769   72                   8               2.0        2.0       110.93                      0              4            0                 144.0        12.325556       1          0    False

## 25 · Interpretation and Business Insight

**Key findings:**
- **Destination risk level** is the strongest claim predictor.
- **Chronic conditions** significantly increase claim probability.
- **Previous claims** history is a strong risk indicator.
- **Age** has a moderate positive effect on claims.
- **Premium plan** holders file fewer claims (healthier/better prepared).

**Business takeaway:** Price premiums based on destination risk, chronic conditions, and claims history. Premium plan holders are lower risk.

## 26 · Limitations

1. Synthetic data with simplified claim dynamics.
2. No claim amount (severity) information.
3. Missing trip purpose (business vs. leisure).
4. No weather or seasonal data.
5. Binary claim ignores partial claims.

## 27 · How to Improve This Project

1. Use real insurance claims data.
2. Add claim severity (amount) as a regression target.
3. Include trip purpose and activity type.
4. Add seasonal and weather features.
5. Model claim frequency and severity separately (actuarial approach).

## 28 · Production Considerations

- Deploy for real-time policy pricing.
- Output claim probability for underwriting.
- Monitor claim rates by destination quarterly.
- Integrate with travel advisory data.
- Calibrate probabilities for reserve estimation.

## 29 · Common Mistakes

1. Using accuracy on 75/25 imbalanced data.
2. Not considering claim severity alongside frequency.
3. Ignoring the seasonal nature of travel.
4. Not calibrating predicted probabilities.
5. Treating all destinations within a risk level equally.

## 30 · Mini Challenge / Exercises

1. Remove `destination_risk` — how much does recall drop?
2. Try SMOTE or class weights for the 75/25 imbalance.
3. Group age into buckets and analyze claim rates.
4. Plot claim rate vs. trip duration.
5. Calculate expected claim cost = P(claim) × avg claim amount.

## 31 · Final Summary / Key Takeaways

1. **Destination risk** is the primary claim driver.
2. **Chronic conditions** and **prior claims** are strong signals.
3. **Age** moderately increases claim risk.
4. **Premium plan** holders are paradoxically lower risk.
5. **Class imbalance** (75/25) requires careful metric selection.
6. **Actuarial models** separate frequency and severity.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Travel Insurance Claim Classification\metrics.json
{
  "CatBoost": {
    "accuracy": 0.7512,
    "f1": 0.2346,
    "precision": 0.5083,
    "recall": 0.1525
  },
  "LightGBM": {
    "accuracy": 0.7488,
    "f1": 0.1796,
    "precision": 0.4889,
    "recall": 0.11
  },
  "Logistic Regression": {
    "accuracy": 0.7588,
    "f1": 0.2548,
    "precision": 0.5593,
    "recall": 0.165
  },
  "FLAML": {
    "accuracy": 0.6931,
    "f1": 0.3356,
    "precision": 0.3658,
    "recall": 0.31
  }
}
